In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder \
    .appName("Week6_PySpark_Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created Successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/25 22:10:02 WARN Utils: Your hostname, Tanvis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.201 instead (on interface en0)
26/07/25 22:10:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/25 22:10:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully


### Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

**Driver:**  
The Driver is the main process of a Spark application. It creates the SparkSession, converts the user's code into tasks, schedules the execution, and collects the results from the executors.

**Cluster Manager:**  
The Cluster Manager is responsible for allocating resources across the cluster. It manages worker nodes and assigns executors to run Spark applications.

**Executor:**  
Executors are worker processes that execute the tasks assigned by the Driver. They process data, perform computations, store intermediate results in memory or disk, and send the results back to the Driver.

### Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

Spark uses Lazy Evaluation, which means transformations are not executed immediately. Instead, Spark records all transformations in a Directed Acyclic Graph (DAG) and executes them only when an action is called.

This allows Spark to optimize the execution plan, combine multiple transformations, reduce unnecessary computations, and minimize disk I/O, resulting in faster processing of large datasets.

In [6]:
# Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 
df = spark.read.csv(
    "data/data.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-------------------+----------+--------------------+--------------------+---------+-----+--------------+---------------+--------------+--------------------+----------------+-----------------+-----------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+-------------+---------+
|           Order No|Order Date|       Customer Name|             Address|     City|State| Customer Type|Account Manager|Order Priority|        Product Name|Product Category|Product Container|  Ship Mode| Ship Date|Cost Price|Retail Price|Profit Margin|Order Quantity|Sub Total|Discount %|Discount $|Order Total|Shipping Cost|    Total|
+-------------------+----------+--------------------+--------------------+---------+-----+--------------+---------------+--------------+--------------------+----------------+-----------------+-----------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+-------------+

### Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

CSV is a row-based file format where all columns of each row are stored together. It is human-readable but generally requires more storage space and slower processing.

Parquet is a columnar storage format where values of the same column are stored together. It provides better compression, faster query performance, and efficient reading of only the required columns, making it ideal for big data processing in Spark.

In [8]:
#Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 
df.filter(col("Product Category") == "Furniture") \
    .select("Order No", "Retail Price") \
    .show()

+-------------------+------------+
|           Order No|Retail Price|
+-------------------+------------+
|5023-01-01 00:00:00|       $2.60|
|5023-01-01 00:00:00|       $2.60|
|5028-01-01 00:00:00|       $2.88|
|5036-01-01 00:00:00|       $1.14|
|5037-01-01 00:00:00|       $8.34|
|5039-01-01 00:00:00|       $7.28|
|5043-01-01 00:00:00|       $3.69|
|5056-01-01 00:00:00|       $2.88|
|5129-01-01 00:00:00|      $19.98|
|5138-01-01 00:00:00|       $2.62|
|5148-01-01 00:00:00|       $2.60|
|5163-01-01 00:00:00|       $2.62|
|5165-01-01 00:00:00|       $2.61|
|5166-01-01 00:00:00|     $165.20|
|5166-01-01 00:00:00|      $20.99|
|5201-01-01 00:00:00|       $5.68|
|5208-01-01 00:00:00|       $1.14|
|5215-01-01 00:00:00|      $19.98|
|5221-01-01 00:00:00|       $1.88|
|5229-01-01 00:00:00|       $5.68|
+-------------------+------------+
only showing top 20 rows


In [10]:
#Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import DoubleType

df_revised = (
    df.withColumnRenamed("Customer Name", "Customer")
      .withColumn(
          "Retail Price",
          regexp_replace(col("Retail Price"), "[$,]", "").cast(DoubleType())
      )
)

df_revised.printSchema()
df_revised.show(5)

root
 |-- Order No: timestamp (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Customer: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Customer Type: string (nullable = true)
 |-- Account Manager: string (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Container: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Cost Price: string (nullable = true)
 |-- Retail Price: double (nullable = true)
 |-- Profit Margin: string (nullable = true)
 |-- Order Quantity: integer (nullable = true)
 |-- Sub Total: string (nullable = true)
 |-- Discount %: string (nullable = true)
 |-- Discount $: string (nullable = true)
 |-- Order Total: string (nullable = true)
 |-- Shipping Cost: string (nullable = true)
 |-- T

### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Spark maintains a Lineage Graph (DAG) that records all transformations applied to a dataset. If a worker node fails and some data is lost, Spark uses the DAG to recompute only the missing partitions instead of recalculating the entire dataset. This provides efficient fault tolerance without requiring data replication after every operation.

In [12]:
#Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import DoubleType

# Clean Order Total and convert it to Double
df_clean = df.withColumn(
    "Order Total",
    regexp_replace(col("Order Total"), "[$,]", "").cast(DoubleType())
)

# Filter records
df_clean.filter(
    (col("Order Priority") == "Critical") &
    (col("Order Total") > 1000)
).show()

+-------------------+----------+------------------+--------------------+---------+-----+--------------+------------------+--------------+--------------------+----------------+-----------------+--------------+----------+----------+------------+-------------+--------------+----------+----------+----------+-----------+-------------+----------+
|           Order No|Order Date|     Customer Name|             Address|     City|State| Customer Type|   Account Manager|Order Priority|        Product Name|Product Category|Product Container|     Ship Mode| Ship Date|Cost Price|Retail Price|Profit Margin|Order Quantity| Sub Total|Discount %|Discount $|Order Total|Shipping Cost|     Total|
+-------------------+----------+------------------+--------------------+---------+-----+--------------+------------------+--------------+--------------------+----------------+-----------------+--------------+----------+----------+------------+-------------+--------------+----------+----------+----------+---------

### Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is an optimization technique in Parquet where filtering conditions are applied while reading the file. Instead of loading the entire dataset into memory, Spark reads only the rows that satisfy the filter condition. This reduces disk I/O, lowers memory usage, and improves query performance.

In [13]:
#Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 
# Add a new column Final Price by applying 18% tax on Retail Price
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn(
    "Retail Price",
    regexp_replace(col("Retail Price"), "[$,]", "").cast("double")
)

df = df.withColumn(
    "Final Price",
    col("Retail Price") * 1.18
)

df.select("Retail Price", "Final Price").show(5)

+------------+------------------+
|Retail Price|       Final Price|
+------------+------------------+
|      300.97|          355.1446|
|        1.26|            1.4868|
|       80.98|           95.5564|
|         8.6|            10.148|
|        2.78|3.2803999999999998|
+------------+------------------+
only showing top 5 rows


### Q11: What is the difference between Transformations and Actions? Provide two examples of each.

**Transformations** are operations that create a new DataFrame or RDD from an existing one. They are lazily evaluated, meaning they are not executed until an action is called.

**Examples of Transformations:**
- `filter()`
- `select()`

**Actions** are operations that trigger the execution of transformations and return a result or write data to storage.

**Examples of Actions:**
- `show()`
- `collect()`

In [17]:
df = spark.read.csv("data/data.csv", header=True, inferSchema=True)

df.write.mode("overwrite").parquet("data/parquet_data")

In [18]:
#Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 
from pyspark.sql.functions import col

df = spark.read.parquet("data/parquet_data")

filtered_df = df.filter(col("Order No").isNotNull())

filtered_df.write.mode("overwrite") \
    .option("header", True) \
    .csv("data/output_csv")

### Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

**Client Mode:**  
In Client Mode, the Driver program runs on the client machine that submits the Spark application. It communicates directly with the cluster manager and executors. This mode is commonly used for development and testing.

**Cluster Mode:**  
In Cluster Mode, the Driver program runs inside the cluster on one of the worker nodes. The cluster manager launches and manages the Driver and Executors. This mode is preferred for production environments because the application continues to run even if the client machine disconnects.

In [19]:
#Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'. 
from pyspark.sql.functions import col

# Filter rows where State is 'North' OR Order Priority is 'High'

df.filter(
    (col("State") == "North") |
    (col("Order Priority") == "High")
).show()

+-------------------+----------+--------------------+--------------------+---------+-----+--------------+----------------+--------------+--------------------+----------------+-----------------+--------------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+-------------+---------+
|           Order No|Order Date|       Customer Name|             Address|     City|State| Customer Type| Account Manager|Order Priority|        Product Name|Product Category|Product Container|     Ship Mode| Ship Date|Cost Price|Retail Price|Profit Margin|Order Quantity|Sub Total|Discount %|Discount $|Order Total|Shipping Cost|    Total|
+-------------------+----------+--------------------+--------------------+---------+-----+--------------+----------------+--------------+--------------------+----------------+-----------------+--------------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+--

### Q15: When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

The `.show(5)` method displays only the first five rows of the dataset, making it fast and memory-efficient for data exploration.

The `.collect()` method retrieves all the data from the cluster to the Driver program. On a multi-terabyte dataset, this can consume a large amount of memory, slow down the application, or even cause the Driver to crash due to an OutOfMemory error.

Therefore, `.show(5)` is the safer choice for previewing large datasets.